# Treinamento: Specialist 910 (Família C - Mag Regional)
**Missão:** Refinamento de ameaça. Recebe exclusivamente as instâncias que sobreviveram aos dois filtros anteriores (Gatekeeper e Great Filter). Seu trabalho é separar o que é impacto moderado (Classe C) do que é impacto severo (Classes M e X).

**Metodologia de Cascata:** O dataset é filtrado pelo modelo `gatekeeper_v1.joblib` e logo em seguida pelo `great_filter_v1.joblib` (Double Filtering) antes do treinamento.

**Estratégia de Punição (Recall-Biased):** Como a topologia magnética isolada possui sobreposição de assinaturas entre C forte e M fraco, o Optuna foi refatorado para forçar pesos matemáticos extremos (`scale_pos_weight`) em favor da classe M/X, e o ajuste de limiar validado através do F4-Score.

## Setup

In [ ]:
import os
import optuna
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
from optuna.integration import XGBoostPruningCallback
import cupy as cp

from src.py_src.models import GatekeeperModel, GreatFilterModel, Specialist910Model
import joblib

In [ ]:
load_dotenv()

REGIONAL_MAG_SLIDED_PATH = os.path.join(os.getenv("SLIDED_PATH"), "mag_regional_slided.parquet")

regional_slided_df = pd.read_parquet(REGIONAL_MAG_SLIDED_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
metadata_cols = ['run_id', 'T_REC_round', 'REGION_ID', 'DATASET_QUERY']

features = [col for col in regional_slided_df.columns if col not in metadata_cols + [target_class, target_flux]]

# Isolamento conceitual das features (X) e do alvo bruto (y)
X = regional_slided_df[features]
y = regional_slided_df[target_class]

In [ ]:
regional_slided_df.head()

## Preparing Data (Group-Based Split - Target M/X)

In [ ]:
cols_to_keep = features + metadata_cols + [target_class, target_flux]
spec910_pool = regional_slided_df[cols_to_keep].copy()

def group_based_chronological_split(df, time_col, region_col, target_col, flux_col, _train_years, _val_years, _test_years):
    df = df.sort_values([region_col, time_col]).reset_index(drop=True).copy()
    
    harp_birth = df.groupby(region_col)[time_col].min().dt.year.to_dict()
    df['harp_birth_year'] = df[region_col].map(harp_birth)

    df['split'] = 'none'
    df.loc[df['harp_birth_year'].isin(_train_years), 'split'] = 'train'
    df.loc[df['harp_birth_year'].isin(_val_years), 'split'] = 'val'
    df.loc[df['harp_birth_year'].isin(_test_years), 'split'] = 'test'

    df = df[df['split'] != 'none'].reset_index(drop=True)

    dict_ = {'x': {}, 'y': {}, 'flux': {}}
    cols_to_drop = [target_col, flux_col, time_col, 'harp_birth_year', 'split'] + metadata_cols

    for split_name in ['train', 'val', 'test']:
        split_df = df[df['split'] == split_name].copy()
        dict_['x'][split_name] = split_df.drop(columns=cols_to_drop, errors='ignore')
        
        # MUDANÇA DE TARGET: Specialist 910 separa < M (Target 0) de >= M (Target 1)
        # No mapeamento: M=4, X=5.
        dict_['y'][split_name] = split_df[target_col].apply(lambda lb: 1 if lb >= 4 else 0)
        dict_['flux'][split_name] = split_df[flux_col]

    return dict_

train_years = [2010, 2011, 2013, 2014, 2015, 2016, 2018, 2019]
val_years = [2012, 2017]
test_years = [2020, 2021, 2022, 2023, 2024]

data = group_based_chronological_split(
    df=spec910_pool,
    time_col='T_REC_round',
    region_col='REGION_ID',
    target_col=target_class,
    flux_col=target_flux,
    _train_years=train_years,
    _val_years=val_years,
    _test_years=test_years
)

print(f"Tamanho do Treino (Original): {len(data['x']['train'])} amostras")
print(f"Tamanho da Validação (Original): {len(data['x']['val'])} amostras")
print(f"Tamanho do Teste (Original): {len(data['x']['test'])} amostras")

## 🛡️ A Cascata Dupla (Double Filtering)

Para treinar o *Specialist 910*, os dados originais devem passar por **duas barreiras**: O Gatekeeper e o Great Filter. Apenas as ameaças que sobreviveram a ambos (Target 0 = Classe C ou resíduo falso de A/B; Target 1 = M/X) serão utilizadas para calibração.

In [ ]:
gatekeeper_path = os.path.join(os.getenv('REGIONAL_MAG_META_MODELS_PATH'), 'gatekeeper_v1.joblib')
gatekeeper = GatekeeperModel.load(gatekeeper_path)

gf_path = os.path.join(os.getenv('REGIONAL_MAG_META_MODELS_PATH'), 'great_filter_v1.joblib')
great_filter = GreatFilterModel.load(gf_path)

print(f"Limiar GK: {gatekeeper.threshold:.4f} | Limiar GF: {great_filter.threshold:.4f}\n")

In [ ]:
def filter_by_cascade(x_raw: pd.DataFrame, y_raw: pd.Series, flux_raw: pd.Series):
    """Filtra o dataset mantendo apenas instâncias aprovadas pelo Gatekeeper E pelo Great Filter."""
    mask_gk = gatekeeper.predict(x_raw) == 1
    mask_gf = great_filter.predict(x_raw) == 1
    
    # Sobreviventes das duas barreiras
    mask = mask_gk & mask_gf
    
    return x_raw[mask].copy(), y_raw[mask].copy(), flux_raw[mask].copy()

X_train_s910, y_train_s910, flux_train_s910 = filter_by_cascade(data['x']['train'], data['y']['train'], data['flux']['train'])
X_val_s910, y_val_s910, flux_val_s910 = filter_by_cascade(data['x']['val'], data['y']['val'], data['flux']['val'])
X_test_s910, y_test_s910, flux_test_s910 = filter_by_cascade(data['x']['test'], data['y']['test'], data['flux']['test'])

def print_funnel_report(name: str, original_y: pd.Series, survived_y: pd.Series):
    orig_total = len(original_y)
    surv_total = len(survived_y)
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    
    orig_pos = (original_y == 1).sum()
    orig_neg = (original_y == 0).sum()
    surv_pos = (survived_y == 1).sum()
    surv_neg = (survived_y == 0).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    ratio_orig = orig_neg / orig_pos if orig_pos > 0 else 0
    ratio_surv = surv_neg / surv_pos if surv_pos > 0 else 0
    
    print(f"🌪️ {name.upper()} CASCADE REPORT (C vs M/X)")
    print(f"   Volume Total:  {orig_total} -> {surv_total} (Redução global de {red_pct:.1f}%)")
    print(f"   Impacto < M (0): {orig_neg} -> {surv_neg} amostras (Resíduo inferior eliminado: {noise_reduction:.1f}%)")
    print(f"   Impacto >= M (1): {orig_pos} -> {surv_pos} amostras (Sinal severo retido: {signal_retention:.1f}%)")
    print(f"   Novo Balanço:  1 Classe M/X para cada {ratio_surv:.1f} Classes C/Inferiores (Original era 1:{ratio_orig:.1f})")
    print("-" * 75)

print("\n--- IMPACTO DA CASCATA SOBRE O SPECIALIST 910 ---\n")
print_funnel_report("Treino", data['y']['train'], y_train_s910)
print_funnel_report("Validação", data['y']['val'], y_val_s910)
print_funnel_report("Teste", data['y']['test'], y_test_s910)

## Discovery Model
Instanciamos a classe `Specialist910Model`. Em conformidade com o comportamento magnético validado na etapa do Great Filter, não limitamos os pesos de instâncias ambíguas (`buffer_limits=None`).

In [ ]:
discovery_model = Specialist910Model(
    buffer_limits=None, # Desativado para a família C (Mag Regional)
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    }
)

In [ ]:
selected_features = discovery_model.discover_top_features(
    x=X_train_s910,
    y=y_train_s910,
    flux_values=flux_train_s910,
    cumulative_threshold=0.95
)

In [ ]:
selected_features

## Hyperparameter Tuning (Optuna Recall-Biased)
A otimização tradicional baseada na distribuição natural falhou em reter as explosões severas de forma precisa. O espaço de busca da variável `scale_pos_weight` foi alterado de `[0.5, 2.0]` para **`[2.0, 6.0] * imbalance_ratio`**, forçando o XGBoost a priorizar a penalização de Falsos Negativos na classe M/X.

In [ ]:
print("Transferindo dados para a VRAM da GPU...")

X_train_filtered = X_train_s910[selected_features].astype('float32')
X_val_filtered = X_val_s910[selected_features].astype('float32')

X_train_gpu = cp.array(X_train_filtered.values)
y_train_gpu = cp.array(y_train_s910.values.astype('float32'))

X_val_gpu = cp.array(X_val_filtered.values)
y_val_gpu = cp.array(y_val_s910.values.astype('float32'))

flux_train_gpu = cp.array(flux_train_s910.values.astype('float32'))
flux_val_gpu = cp.array(flux_val_s910.values.astype('float32'))

print("Transferência concluída. Dados alocados na GPU.")

In [ ]:
def objective(trial):
    neg_count = (y_train_s910 == 0).sum()
    pos_count = (y_train_s910 == 1).sum()
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-aucpr')

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'device': 'cuda',

        'early_stopping_rounds': 50,
        'callbacks': [pruning_callback],
        
        'max_delta_step': trial.suggest_int('max_delta_step', 1, 10),
        
        # O CÉTICO: Pesos de classe próximos ao balanço natural.
        'scale_pos_weight': trial.suggest_float("scale_pos_weight", imbalance_ratio * 0.5, imbalance_ratio * 1.5),
        
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 25)
    }

    model = Specialist910Model(
        params=params, 
        buffer_limits=None, 
        features_to_keep=None
    )

    model.fit(
        x=X_train_gpu,
        y=y_train_gpu,
        flux_values=flux_train_gpu,
        eval_set=[(X_val_gpu, y_val_gpu)],
        verbose=False
    )

    y_pred_proba_raw = model.predict_proba(X_val_gpu)[:, 1]
    y_pred_proba_cpu = y_pred_proba_raw.get() if hasattr(y_pred_proba_raw, 'get') else y_pred_proba_raw

    pr_auc = average_precision_score(y_val_s910, y_pred_proba_cpu)

    return pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning com viés agressivo em Recall...")
study.optimize(objective, n_trials=500)

print(f"\nBest Score (PR AUC): {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'aucpr', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

In [ ]:
final_model = Specialist910Model(
    params=study.best_params, 
    buffer_limits=None,
    features_to_keep=selected_features
)

final_model.fit(
    x=X_train_s910, 
    y=y_train_s910,
    flux_values=flux_train_s910,
    verbose=True
)

## ⚖️ Trade-off de Decisão (Threshold Tuning)

Uma vez que o `beta=2.0` (F2-Score) barrou o abaixamento do limiar porque a Precisão colapsou, estamos dobrando a tolerância à queda de precisão ao adotar `beta=4.0` (F4-Score). O comando matemático agora é claro: "Aceite o aumento massivo de Falsos Positivos da Classe C, desde que possamos resgatar as Classes M e X."

In [ ]:
fig = final_model.get_threshold_graph(X_val_s910, y_val_s910)
# display(fig)

In [ ]:
optimal_threshold = final_model.optimize_threshold(
    X_val_s910,
    y_val_s910,
    target_recall=None
)

## Results

In [ ]:
x_test_df = X_test_s910
y_true = y_test_s910.values.astype(int)
flux_test = flux_test_s910

y_prob = final_model.predict_proba(x_test_df)[:, 1]
y_pred = (y_prob >= optimal_threshold).astype(int)

# Conforme metodologia estabelecida para a Tabela Regional, persistência é estrita zero.
y_persistence = np.zeros_like(y_true)

In [ ]:
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(final_model.get_classification_report(y_true, y_pred, target_names=['Impacto Moderado (< M)', 'Ameaça Severa (M/X)']))

In [ ]:
print("\n--- MÉTRICAS ABRANGENTES ---")
comprehensive_df = final_model.get_comprehensive_metrics(y_true, y_pred, y_prob)
display(comprehensive_df)

In [ ]:
print("\n--- PR-F1 (SKILL SCORE RELATIVO) ---")
pr_f1_score = final_model.calculate_prss(y_true, y_pred, y_persistence)
print(f"PR-F1 Score: {pr_f1_score:.4f}")

In [ ]:
print("\n--- ANÁLISE AC/NC (ACTIVITY CHANGE) ---")
ac_nc_df = final_model.analyze_ac_nc_performance(y_true, y_pred, y_persistence)
display(ac_nc_df)

In [ ]:
print("\n--- DISTRIBUIÇÃO DE ERROS POR CLASSE SOLAR ---")
error_dist_df = final_model.analyze_error_distribution(y_true, y_pred, flux_test)
display(error_dist_df)

In [ ]:
print("\n--- ANÁLISE DE FLUXO (ZONAS) ---")
fig_flux, summary_flux = final_model.analyze_flux_errors(y_true, y_pred, flux_test, buffer_limits=[1e-5, 1e-4])
display(summary_flux)
display(fig_flux)

## Features Importance

In [ ]:
print("\n--- IMPORTÂNCIA DAS FEATURES (GAIN) ---")
features_importance = final_model.get_feature_importance()
display(features_importance.head(10))

## Export

In [ ]:
SAVE_PATH = os.getenv('REGIONAL_MAG_META_MODELS_PATH')
os.makedirs(SAVE_PATH, exist_ok=True)

final_model.save(os.path.join(SAVE_PATH, 'specialist_910_v1.joblib'))

print(f"Modelo Specialist 910 exportado com sucesso para: {SAVE_PATH}")
print(f"Threshold otimizado embutido: {final_model.threshold:.4f}")
print(f"Total de features retidas: {len(final_model.features_to_keep)}")